# 命名實體辨識（NER）：用 BERT 找出句子裡的人名、組織、地點

---

## 這份程式要做什麼

給一句話，標出裡面的**實體**（人名、組織、地點…）：

```
輸入：  My name is Maarten and I live in Amsterdam .
輸出：  My name is [Maarten: 人名] and I live in [Amsterdam: 地點] .
```

這叫 **NER（Named Entity Recognition，命名實體辨識）**。

## 它屬於哪一種任務：Token Classification

| 任務 | 每個「什麼」一個標籤 | 例子 |
|---|---|---|
| 序列分類（影評情緒） | 一**句**一個標籤 | 這句影評是正面還是負面 |
| **Token 分類（NER，這份）** | 一**個詞**一個標籤 | 這個詞是人名、地名，還是都不是 |
| 遮蔽語言模型（MLM） | 被遮住的位置預測一個詞 | 填空 |

所以模型要對**每個 token** 輸出一個標籤，而不是整句一個。

## NER 的標籤長什麼樣：BIO 標記法

| 標記 | 意思 |
|---|---|
| `B-xxx`（Begin） | 一個實體的**開頭** |
| `I-xxx`（Inside） | 同一個實體的**延續** |
| `O`（Outside） | **不是實體** |

```
Dean   Palmer  hit  his  30th  homer  for  the  Rangers  .
B-PER  I-PER   O    O    O     O      O    O    B-ORG    O
```

`Dean Palmer` 是同一個人名：`Dean` 是開頭（B），`Palmer` 是延續（I）。

## 整體流程

```
載入 CoNLL-2003 → 建立標籤對照 → 載入 BERT（Token 分類版）
   → 分詞並對齊標籤（最關鍵、最難懂的一步）
   → 設定評估指標（seqeval）→ 訓練 → 評估 → 存檔 → 用 pipeline 預測
```

## 與前面學過的方法比較

| | 影評分類 | **NER（這份）** | MLM |
|---|---|---|---|
| 模型類別 | `...ForSequenceClassification` | `...ForTokenClassification` | `...ForMaskedLM` |
| 標籤粒度 | 一句一個 | **一個 token 一個** | 被遮住處 |
| collator | `DataCollatorWithPadding` | `DataCollatorForTokenClassification` | `DataCollatorForLanguageModeling` |
| 指標 | F1 | **seqeval（實體層級 F1）** | loss / 困惑度 |

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `load_dataset` | 載入資料集 |
| `AutoModelForTokenClassification` | 「BERT + Token 分類頭」，每個 token 輸出一組類別分數 |
| `AutoTokenizer` | 配對的分詞器 |
| `evaluate` | 載入評估指標（這裡用 `seqeval`） |
| `TrainingArguments`、`Trainer` | 設定並執行訓練 |
| `DataCollatorForTokenClassification` | 組 batch 時，同時補齊輸入**和標籤** |
| `pipeline` | 訓練後一行程式做預測 |


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
import evaluate
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
from transformers import pipeline

## 步驟 1：載入 CoNLL-2003 資料集

CoNLL-2003 是 NER 的經典英文資料集，來源是新聞文章。每筆資料是**一句話**，包含：

| 欄位 | 說明 |
|---|---|
| `tokens` | 這句話切成的**詞**（已經切好，是 list of strings） |
| `ner_tags` | 每個詞的 NER 標籤（**數字編號**） |
| `pos_tags`、`chunk_tags` | 詞性、片語標籤（這份程式用不到） |

train 1.4 萬句、validation 3250 句、test 3453 句，請用 `len(dataset["train"])` 確認。

`trust_remote_code=True`：這個資料集是用「載入腳本」提供的，需要允許執行該腳本。較新版本的 `datasets` 對腳本型資料集的支援有變動，如果你在別的環境載入失敗，優先檢查版本。


In [ ]:
dataset = load_dataset("eriktks/conll2003", trust_remote_code=True)
# print(dataset)
# example = dataset["train"][848]
# print(example)

## 步驟 2：建立標籤對照表

```python
label2id = {label: i for i, label in enumerate(dataset["train"].features["ner_tags"].feature.names)}
id2label = {i: label for label, i in label2id.items()}
```

資料裡的標籤是**數字**，但我們也想看得懂名稱，所以要建立兩張對照表：

| 變數 | 方向 | 例子 |
|---|---|---|
| `label2id` | 名稱 → 編號 | `{'O': 0, 'B-PER': 1, ...}` |
| `id2label` | 編號 → 名稱 | `{0: 'O', 1: 'B-PER', ...}` |

### 這兩行怎麼讀

1. `dataset["train"].features["ner_tags"].feature.names`：取出標籤名稱的 list，依序就是編號 0, 1, 2...
2. `enumerate(...)`：把 list 變成 `(編號, 名稱)` 的配對
3. `{label: i for i, label in ...}`：**字典生成式**，建立「名稱 → 編號」
4. 第二行把鍵和值**對調**，就得到「編號 → 名稱」

### CoNLL-2003 的 9 個標籤

| 編號 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| 標籤 | O | B-PER | I-PER | B-ORG | I-ORG | B-LOC | I-LOC | B-MISC | I-MISC |

（請用註解掉的 `print(id2label)` 確認；PER = 人名、ORG = 組織、LOC = 地點、MISC = 其他。）

**請注意這個規律：B 都是奇數、I 都是偶數、O 是 0。** 後面「對齊標籤」那一步會用到這個規律。

In [ ]:
label2id = {label: i for i, label in enumerate(dataset["train"].features["ner_tags"].feature.names)}
id2label = {i: label for label, i in label2id.items()}
# print(label2id)
# print(id2label)

## 步驟 3：載入預訓練模型與分詞器

```python
model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)
```

| 參數 | 說明 |
|---|---|
| `bert-base-cased` | **區分大小寫**的 BERT。NER 很吃大小寫（`Apple` 公司 vs `apple` 水果；人名地名通常首字大寫），所以選 cased 版 |
| `num_labels=9` | 分類頭要輸出 9 類 |
| `id2label` / `label2id` | 把對照表存進模型的設定，之後 pipeline 才能直接輸出 `B-PER` 這種名稱，而不是 `LABEL_1` |

和之前一樣：BERT 本體是**預訓練好的權重**，最上面的 Token 分類頭是**新加的、隨機初始化**的，載入時看到 "newly initialized" 的警告是正常的。

### 它和序列分類模型有什麼不同

| | 序列分類 | Token 分類 |
|---|---|---|
| 取用的輸出 | 只用 `[CLS]` 位置的向量 | **每一個 token** 的向量 |
| 輸出形狀 | `(句數, 類別數)` | `(句數, 序列長度, 類別數)` |

In [ ]:
model_id = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)

## 步驟 4：先弄懂問題 —— 詞和 token 對不上

下面這個儲存格是你自己留下的**探索紀錄**（都是註解），記錄了 848 號資料實際的輸出。這是理解後面最難的函式的關鍵，請仔細看。

```
詞（tokens）：   Dean  Palmer  hit  his  30th  homer  for  the  Rangers  .        → 10 個詞
ner_tags：       [1,    2,     0,   0,   0,    0,     0,   0,   3,       0]       → 10 個標籤

分詞後（sub_tokens）：
[CLS] Dean Palmer hit his 30th home ##r for the Rangers . [SEP]                   → 13 個 token

word_ids：
[None,  0,   1,    2,   3,   4,    5,   5,  6,   7,   8,     9,  None]
```

### 為什麼對不上

| 差異 | 原因 |
|---|---|
| 多了 `[CLS]`、`[SEP]` | BERT 會自動在句首句尾加上特殊 token |
| `homer` 變成 `home` + `##r` | 詞不在詞彙表時，分詞器會切成子詞（subword），`##` 表示是前一塊的延續 |

模型會輸出 13 個預測，但你只有 10 個答案，**無法計算 loss**。所以需要「對齊」。

### `word_ids` 是對齊的線索

`word_ids` 告訴每個 token **來自第幾個詞**：

- `None`：特殊 token（`[CLS]`、`[SEP]`）
- 兩個 token 的數字相同（例如兩個 `5`）：它們是**同一個詞被切開**的

In [ ]:
# token_ids = tokenizer(example["tokens"], is_split_into_words = True)
# print(token_ids)
# word_ids = token_ids.word_ids()
# print(word_ids)



# token_ids2 = tokenizer(example["tokens"], is_split_into_words = True)["input_ids"]
# print(token_ids2)
# sub_tokens = tokenizer.convert_ids_to_tokens(token_ids2)
# print(sub_tokens)



#=====================================================================
#'ner_tags': [1, 2, 0, 0, 0, 0, 0, 0, 3, 0]}
# token_ids : {'input_ids': [101, 4285, 8450, 1855, 1117, 13631, 1313, 1197, 1111, 1103, 6838, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
# word_ids : [None, 0, 1, 2, 3, 4, 5, 5, 6, 7, 8, 9, None]
# token_ids["input_ids"] : [101, 4285, 8450, 1855, 1117, 13631, 1313, 1197, 1111, 1103, 6838, 119, 102]
# sub_tokens : ['[CLS]', 'Dean', 'Palmer', 'hit', 'his', '30th', 'home', '##r', 'for', 'the', 'Rangers', '.', '[SEP]']

## 步驟 5：`align_labels` —— 把「詞的標籤」攤到「token 的位置」

這是整份程式**最難、也最關鍵**的函式。目標：把 10 個標籤變成 13 個。

### 規則

| token 的情況 | 給什麼標籤 |
|---|---|
| 特殊 token（`word_id` 是 `None`） | `-100`（PyTorch 計算 loss 時會**忽略**） |
| 一個詞的**第一塊** | 該詞原本的標籤 |
| 一個詞被切開後的**後半段** | 沿用該詞的標籤；但如果原標籤是 `B-`，要改成 `I-` |

### 三個分支各自負責什麼

```python
if word_idx != previous_word_idx:     # ① 和上一個 token 不同詞 → 新詞的第一塊（或結尾的 [SEP]）
elif word_idx is None:                # ② word_id 是 None 且和上一個一樣 → 開頭的 [CLS]
else:                                 # ③ 和上一個 token 同一個詞 → 被切開的後半段
```

`previous_word_idx` 的作用：**和上一個 token 比較，判斷現在是新詞還是同一個詞的延續。**

`[CLS]` 會走②，是因為 `previous_word_idx` 一開始是 `None`，`None != None` 為 `False`，進不了①。結尾的 `[SEP]` 因為前一個是 `9`，所以走①。

### 逐 token 帶入你的 848 號資料

`label = [1, 2, 0, 0, 0, 0, 0, 0, 3, 0]`（以下由我用純 Python 複製這段邏輯，帶入你貼的真實輸出跑出）

| token | word_id | 分支 | 加入的標籤 |
|---|---|---|---|
| `[CLS]` | None | ② | -100 |
| Dean | 0 | ① | 1 |
| Palmer | 1 | ① | 2 |
| hit | 2 | ① | 0 |
| his | 3 | ① | 0 |
| 30th | 4 | ① | 0 |
| home | 5 | ① | 0 |
| `##r` | 5 | ③ | 0 |
| for | 6 | ① | 0 |
| the | 7 | ① | 0 |
| Rangers | 8 | ① | 3 |
| . | 9 | ① | 0 |
| `[SEP]` | None | ① | -100 |

結果：`[-100, 1, 2, 0, 0, 0, 0, 0, 0, 0, 3, 0, -100]`，13 個，和 token 數一致。

### `% 2 == 1` 的用途：B 變 I

848 號資料沒有「被切開的實體詞」，看不到這個效果。再看一個**假設**的例子（切法是我假設的，實際以 tokenizer 為準）：

```
句子：Rangers beat Mariners      標籤：[3, 0, 3]
假設 Mariners 被切成：Mari  ##ners
tokens：   [CLS]  Rangers  beat  Mari  ##ners  [SEP]
word_ids： None     0       1     2      2     None
```

| token | 分支 | 加入的標籤 |
|---|---|---|
| `[CLS]` | ② | -100 |
| Rangers | ① | 3 |
| beat | ① | 0 |
| Mari | ① | 3 |
| `##ners` | ③ | 3 是奇數 → +1 → **4** |
| `[SEP]` | ① | -100 |

`Mari` 是 `B-ORG`（3），後半段 `##ners` 被改成 `I-ORG`（4）。這符合 BIO 規則：實體的第一塊用 B，後續用 I。

這招之所以行得通，就是因為前面提過的規律：**B 是奇數、I 是偶數**，所以 `% 2 == 1` 就能判斷「原標籤是 B」，`+1` 就能轉成對應的 I。

| 被切開的詞原本的標籤 | 判斷 | 後半段得到 |
|---|---|---|
| `B-ORG`（3，奇數） | `% 2 == 1`，+1 | `I-ORG`（4） |
| `I-PER`（2，偶數） | 不變 | `I-PER`（2） |
| `O`（0，偶數） | 不變 | `O`（0） |

另一種常見做法是「只給第一塊標籤，後半段全部給 -100」，兩種都有人用。這份程式是前者：**每一塊都給標籤**。

### 為什麼函式裡有 `for index, label in enumerate(labels)`

這個函式是給 `map(..., batched=True)` 用的，所以 `example["tokens"]` 是**很多句**（list of lists）：

| 變數 | 意思 |
|---|---|
| `index` | 這一批裡的第幾句 |
| `label` | 那一句的標籤 list |
| `token_ids.word_ids(batch_index = index)` | 取出那一句的 `word_ids` |

### 最後兩行很重要

```python
token_ids["labels"] = updated_labels
return token_ids
```

把算好的標籤放進 `labels` 欄位再回傳。**沒有這一行，資料集就沒有答案，Trainer 無法計算 loss。**

### 為什麼可以對「不存在的 key」賦值

分詞器回傳的 `token_ids` 裡，原本只有 `input_ids`、`token_type_ids`、`attention_mask`，並沒有 `labels`。但 `token_ids["labels"] = updated_labels` 是**寫入**，不是讀取：

| 操作 | 寫法 | key 不存在時 |
|---|---|---|
| 讀取 | `d["labels"]` | 報 `KeyError` |
| 寫入 | `d["labels"] = 值` | **直接新增這個 key** |

我用 `BatchEncoding`（分詞器回傳的物件型別）在沙箱裡實際執行的結果：

```
寫入前的 key： ['input_ids', 'token_type_ids', 'attention_mask']
讀取不存在的 key → KeyError: 'labels'
寫入後的 key： ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
```

`token_ids` 是 `BatchEncoding` 物件，不是普通 dict，但用起來像 dict，所以可以這樣新增欄位。

**欄位名稱一定要叫 `labels`**：Trainer 訓練時會把欄位整包傳給模型，模型的 `forward` 有一個參數就叫 `labels`，收到它才會計算 loss。我沒有針對你的版本確認 Trainer 對其他名稱是否有自動處理。

`ner_tags`（詞層級，10 個）和 `labels`（token 層級，13 個）是兩個不同的欄位。

In [ ]:
def align_labels(example):
    token_ids = tokenizer(
        example["tokens"],
        truncation = True,
        is_split_into_words = True,
    )
    labels = example["ner_tags"]

    updated_labels = []
    for index, label in enumerate(labels):
        word_ids = token_ids.word_ids(batch_index = index)
        previous_word_idx = None

        label_ids = []
        for word_idx in word_ids:
            if word_idx != previous_word_idx:
                previous_word_idx = word_idx
                updated_label = -100 if word_idx is None else label[word_idx]
                label_ids.append(updated_label)

            elif word_idx is None:
                label_ids.append(-100)

            else: 
                updated_label = label[word_idx]
                if updated_label % 2 == 1:
                    updated_label += 1
                label_ids.append(updated_label)

        updated_labels.append(label_ids)
    token_ids["labels"] = updated_labels
    return token_ids

## 步驟 6：把對齊函式套用到整個資料集

```python
tokenized = dataset.map(align_labels, batched = True)
```

- `map` 把 `align_labels` 套用到 train、validation、test 三份資料
- **`batched = True` 一定要加**：函式裡假設輸入是「很多句」。如果漏掉，傳進來的是單句，`ner_tags` 會變成整數 list，迴圈拿到的 `label` 變成單一整數，`label[word_idx]` 就會報錯
- 處理後的資料集多了 `input_ids`、`attention_mask`、`labels` 等欄位

### 驗證對齊有沒有做對（強烈建議自己跑一次）

```python
ex = tokenized["train"][848]
print(tokenizer.convert_ids_to_tokens(ex["input_ids"]))
print(ex["labels"])     # 預期 [-100, 1, 2, 0, 0, 0, 0, 0, 0, 0, 3, 0, -100]
```

原本的欄位（`tokens`、`ner_tags` 等）還留在資料集裡，但 `Trainer` 預設會自動丟掉模型用不到的欄位（這是我依預設行為推斷的，沒有針對你的版本確認）。

In [ ]:
tokenized = dataset.map(align_labels, batched = True)

## 步驟 7：定義評估指標（seqeval + compute_metrics）

NER 不能只看「逐 token 對不對」，要看**實體有沒有找對**。這就是 `seqeval` 的工作。

### seqeval 做什麼

它先把 BIO 標籤序列**還原成實體**，再比對「類型」和「範圍」是否**完全一致**：

```
正確答案：  PER（Dean Palmer）   ORG（Rangers）
模型預測：  PER（Dean Palmer）   LOC（Rangers）     ← 範圍對但類型錯，整個實體算錯
```

### `compute_metrics` 四步驟

下面是我用一個 2 句話的小 batch，照這段邏輯跑出來的結果（模型故意犯一個錯：句1 的 `Rangers` 真實是 `B-ORG`，預測成 `B-LOC`）。

**① `argmax`：分數變成預測類別**

```python
predictions = predictions.argmax(axis=-1)
```

形狀從 `(句數, 序列長度, 9)` 變成 `(句數, 序列長度)`。`axis=-1` 是在每個 token 的 9 個分數之間選最大的。

**② 兩個 list comprehension：濾掉 -100，數字換成名稱**

```python
[id2label[p] for (p, l) in zip(prediction, label) if l != -100]
```

等價於巢狀迴圈：

```python
for prediction, label in zip(predictions, labels):      # 一句一句
    for p, l in zip(prediction, label):                  # 一個 token 一個 token
        if l != -100:                                    # 要算分才保留
            sent.append(id2label[p])
```

`true_predictions` 放預測、`true_labels` 放正確答案，**兩者都用 `l != -100` 濾**，所以長度一定相同。`[CLS]`、`[SEP]`、補齊的位置都被濾掉。

```
true_predictions:  ['B-PER','I-PER','O','O','O','O','O','O','O','B-LOC','O']
true_labels:       ['B-PER','I-PER','O','O','O','O','O','O','O','B-ORG','O']
```

**③ seqeval 比對實體**

```
正確答案有 4 個實體，模型預測了 4 個，其中 3 個完全一致
precision = 3/4 = 0.75   （模型說有的，有多少是對的）
recall    = 3/4 = 0.75   （真正有的，找出多少）
F1        = 0.75
accuracy  = 14/15 = 0.9333   （逐 token 算，15 個算分的 token 對 14 個）
```

**④ 回傳 dict**

挑出整體（`overall_`）的四個數字。`Trainer` 會自動加上 `eval_` 前綴，所以輸出裡看到的是 `eval_f1`。

### 實體層級有多嚴格

假設模型把 `Palmer` 預測成 `O`：預測的 `PER（Dean）` 和答案的 `PER（Dean Palmer）` 範圍不同，**整個實體算錯**。同一份範例的 F1 會從 0.75 掉到 0.5，而 accuracy 只從 0.933 掉到 0.867。

### 為什麼 accuracy 通常比 F1 高很多

句子裡大部分 token 都是 `O`，全猜 `O` 也能拿高分。所以 NER 主要看 **F1**，accuracy 只當參考。

> 變數名稱 `seqqeval` 多了一個 q，是筆誤，但前後拼法一致，所以不影響執行。

In [ ]:
seqqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)

    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 步驟 8：資料整理器（DataCollatorForTokenClassification）

組 batch 時，同一批的句子長度不同，需要補齊。和影評分類不同的是，NER 有**每個 token 的標籤**，所以不只要補齊輸入，**標籤也要補齊**：

| 項目 | 補什麼 |
|---|---|
| `input_ids` | 補 `[PAD]` |
| `attention_mask` | 補 0（告訴模型這裡是填充） |
| `labels` | 補 **-100**（loss 會忽略） |

一般的 `DataCollatorWithPadding` 只補輸入，不會處理 `labels` 的長度，所以 Token 分類要用專門的版本。labels 補 -100 這點是我依實作的印象寫的，沒有針對你的版本確認。

這也是 `compute_metrics` 裡濾掉 `-100` 的原因之一：補齊出來的位置也是 -100。

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

## 步驟 9：訓練參數

| 參數 | 值 | 說明 |
|---|---|---|
| `output_dir` | `"NER_Finetune_model"` | 輸出資料夾 |
| `learning_rate` | `2e-5` | 小學習率，微調只做小幅調整 |
| `per_device_train_batch_size` | 16 | 訓練時每步 16 句 |
| `per_device_eval_batch_size` | 16 | 評估時每次 16 句 |
| `num_train_epochs` | 1 | 整份訓練資料看 1 遍 |
| `weight_decay` | 0.01 | 輕微的正則化 |
| `save_strategy` | `"epoch"` | 每個 epoch 存一次 checkpoint |
| `report_to` | `"none"` | 不傳記錄到外部工具 |

步數速算：約 1.4 萬句 ÷ 16 ≈ 880 步。

In [ ]:
training_args = TrainingArguments(
    output_dir = "NER_Finetune_model",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 1,
    weight_decay = 0.01,
    save_strategy = "epoch",
    report_to = "none",
)

## 步驟 10：建立 Trainer

| 參數 | 說明 |
|---|---|
| `model` | BERT + Token 分類頭 |
| `args` | 訓練參數 |
| `train_dataset` | 對齊標籤後的訓練集 |
| `eval_dataset` | 對齊標籤後的**測試集** |
| `data_collator` | 補齊輸入與標籤 |
| `compute_metrics` | 剛剛定義的 seqeval 指標函式 |

> **建立 Trainer 不會自動開始訓練**，要呼叫 `trainer.train()`。

In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized["train"],
    eval_dataset = tokenized["test"],
    data_collator = data_collator,
    compute_metrics = compute_metrics,
)

## 步驟 11：訓練、評估、儲存

```python
trainer.train()
result = trainer.evaluate()
print(result)
tokenizer.save_pretrained("NER_Finetune_model")
model.save_pretrained("NER_Finetune_model")
```

| 步驟 | 說明 |
|---|---|
| `trainer.train()` | **真正的微調**，更新模型全部權重 |
| `trainer.evaluate()` | 在測試集上評估，回傳字典 |
| `tokenizer.save_pretrained(...)` | **一定要存**：模型和分詞器是一對，之後從資料夾載入模型時，要在同一個資料夾找到分詞器 |
| `model.save_pretrained(...)` | 存訓練後的模型權重 |

### 你自己跑出來的結果

```
eval_loss: 0.1393   eval_precision: 0.8614   eval_recall: 0.8991
eval_f1: 0.8798     eval_accuracy: 0.9679
```

| 指標 | 值 | 解讀 |
|---|---|---|
| accuracy | 0.968 | 很高，但大部分 token 是 `O`，參考價值有限 |
| precision | 0.861 | 模型標出的實體，約 86% 是對的 |
| recall | 0.899 | 真正的實體，約 90% 被找出 |
| **F1** | **0.880** | NER 的主要指標 |

只訓練 1 個 epoch 就有這個水準。多訓練幾個 epoch 通常還能再進步（我沒有實測，建議自己試）。

### 經典陷阱：只存模型、沒存分詞器

如果漏掉 `tokenizer.save_pretrained(...)`，之後用 `pipeline` 從資料夾載入時，可能找不到正確的分詞器，預測結果會是 **`[]`（空 list）**。`[]` 不是空字串，而是「沒有找到任何實體」：pipeline 預設會濾掉標籤為 `O` 的 token，如果所有 token 都被預測成 `O`（或輸入因為分詞器不對而變成亂碼），結果就是空 list。

In [ ]:
trainer.train()
result = trainer.evaluate()
print(result)
tokenizer.save_pretrained("NER_Finetune_model") 
model.save_pretrained("NER_Finetune_model")

## 步驟 12：用 pipeline 預測新句子

```python
token_classifier = pipeline(
    "token-classification",
    model = "NER_Finetune_model",
    tokenizer = "NER_Finetune_model",
)
```

`pipeline` 是現成的推論工具，`model` 和 `tokenizer` 都可以是**資料夾路徑**，它會自己載入。這裡兩個都指向剛存好的資料夾，所以分詞器和模型一定配對。

### 輸出長什麼樣

回傳的是一個 list，每個元素是一個被認為是實體的 token，格式大致如下（數值只是示意）：

```
[{'entity': 'B-PER', 'score': 0.99, 'index': 4, 'word': 'Maarten', 'start': 11, 'end': 18},
 {'entity': 'B-LOC', 'score': 0.99, 'index': 9, 'word': 'Amsterdam', 'start': 33, 'end': 42}]
```

| 欄位 | 意思 |
|---|---|
| `entity` | 預測的標籤 |
| `score` | 信心分數 |
| `word` | 這個 token 的文字 |
| `start` / `end` | 在原句中的字元位置 |

注意：`O` 的 token **不會**出現在結果裡。

### 想讓輸出更好讀

預設會把每個 token 分開列出，一個被切開的名字會變成好幾筆。加上 `aggregation_strategy` 可以把同一個實體合併：

```python
pipeline("token-classification", model="NER_Finetune_model",
         tokenizer="NER_Finetune_model", aggregation_strategy="simple")
```

這個參數名稱是依我對 transformers 的印象寫的，沒有針對你的版本確認。

In [ ]:
token_classifier = pipeline(
    "token-classification",
    model = "NER_Finetune_model",
    tokenizer = "NER_Finetune_model",
)

pred = token_classifier("My name is Maarten and I live in Amsterdam.")
print(pred)

---

# 重點整理

## 一、整份程式一句話

> 載入預訓練 BERT 並接上 Token 分類頭，用 CoNLL-2003 的新聞句子微調一個 epoch；關鍵是先把「一個詞一個」的標籤，依 `word_ids` 對齊到「一個 token 一個」的位置，再用 seqeval 以實體為單位評估。

## 二、必須搞懂的 10 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | NER | 找出句子中的人名、組織、地點等實體 |
| 2 | Token 分類 | 每個 token 一個標籤，不是整句一個 |
| 3 | BIO 標記 | B = 實體開頭，I = 實體延續，O = 非實體 |
| 4 | 子詞（subword） | 一個詞被切成多塊，後半以 `##` 開頭 |
| 5 | `word_ids` | 每個 token 來自第幾個詞；`None` 是特殊 token |
| 6 | `-100` | loss 忽略的標籤，用在特殊 token 和補齊 |
| 7 | 標籤對齊 | 把詞的標籤攤到 token；被切開的 B 要改成 I |
| 8 | `% 2 == 1` | 因為 B 是奇數、I 是偶數，用來判斷並轉換 |
| 9 | seqeval | 以「實體」為單位比對（類型與範圍都要完全一致） |
| 10 | 分詞器要一起存 | 模型與分詞器是一對，只存模型會導致 pipeline 失效 |

## 三、程式流程速記

```
1. load_dataset                    → CoNLL-2003（tokens + ner_tags）
2. label2id / id2label             → 標籤名稱與編號對照
3. AutoModelForTokenClassification → BERT + 新的 9 類分類頭
4. align_labels + map(batched)     → 分詞、對齊標籤、加入 labels 欄位
5. seqeval + compute_metrics       → argmax → 濾 -100 → 實體層級 P/R/F1
6. DataCollatorForTokenClassification → 補齊輸入與標籤
7. TrainingArguments + Trainer     → 組裝訓練流程
8. train → evaluate → 存 tokenizer 與 model
9. pipeline("token-classification") → 預測新句子
```

## 四、最容易搞混的地方

1. **為什麼不能直接把 `ner_tags` 當 `labels`？**
   詞的數量和 token 的數量不同（多了 `[CLS]`、`[SEP]`，還有被切開的詞），長度對不上。

2. **為什麼要有 -100？**
   特殊 token 和補齊的位置沒有真正的標籤，不該參與 loss 計算，PyTorch 看到 -100 會自動忽略。

3. **被切開的詞，後半段為什麼 B 要變 I？**
   BIO 規則：一個實體只有第一塊是 B，後續都是 I。

4. **為什麼 `align_labels` 要 `batched=True`？**
   函式內假設輸入是很多句，用 `index` 和 `batch_index` 一句一句處理。

5. **為什麼 accuracy 很高、F1 卻低一截？**
   大部分 token 是 `O`，全猜 `O` 也能拿高分；F1 以實體為單位，嚴格得多。

6. **為什麼 pipeline 回傳 `[]`？**
   不是空字串，是「沒找到任何實體」。常見原因是分詞器沒存或不配對，導致模型拿到不正確的輸入。

7. **`compute_metrics` 為什麼要濾 -100？**
   那些位置沒有真正的標籤，不該算分，濾掉才不會稀釋或扭曲分數。

## 五、常見錯誤排查表

| 症狀 | 可能原因 | 檢查方式 |
|---|---|---|
| 訓練時說找不到 labels 或 loss 是 None | `align_labels` 沒有把 `labels` 放進回傳值 | `print(tokenized["train"][0].keys())` 應包含 `labels` |
| `label[word_idx]` 報錯 | 漏了 `batched = True` | 確認 `map(..., batched = True)` |
| 長度不一致的錯誤 | 標籤沒對齊，數量和 token 不同 | 比較 `len(ex["input_ids"])` 和 `len(ex["labels"])` |
| pipeline 回傳 `[]` | 分詞器沒存或不配對 | 檢查資料夾裡有沒有分詞檔；先用 `bert-base-cased` 的分詞器試試 |
| 輸出的標籤是 `LABEL_1` 而不是 `B-PER` | 載入模型時沒傳 `id2label` | 確認 `from_pretrained(..., id2label=..., label2id=...)` |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

1. **評估用的是 test 集**
   資料集有 `validation` 集。標準做法是訓練中用 validation 調整，test 留到最後只看一次。

2. **訓練中不會評估**
   沒設定 `eval_strategy`，只有訓練結束後的那次 `evaluate()`，看不到訓練過程中的進步。

3. **只訓練 1 個 epoch**
   對 NER 來說通常偏少，可以試 3 個 epoch 看 F1 是否上升。

4. **評估是在「子詞 token」層級**
   因為每個子詞都有標籤，`compute_metrics` 是對所有子詞做 seqeval，和「只標第一塊」的做法分數不完全可以直接比較。（這是我依做法推論的，沒有實測差異。）

5. **`output_dir` 與最終存檔同一個資料夾**
   checkpoint 子資料夾和最終模型放在一起，能用，只是較雜；記得把這個資料夾加進 `.gitignore`。

6. **沒有固定隨機種子**
   分類頭是隨機初始化的，每次訓練結果略有不同。

7. **`trust_remote_code=True`**
   需要資料集腳本的支援，環境版本變動時可能失效。

8. **變數名稱筆誤 `seqqeval`**
   不影響執行。

9. **多餘的 import `from cProfile import label`**
   編輯器自動補上的，不影響執行，但建議刪掉。

## 七、可以自己做的實驗（需要時再自己加，不影響原程式）

```python
# 1. 驗證對齊
ex = tokenized["train"][848]
print(tokenizer.convert_ids_to_tokens(ex["input_ids"]))
print(ex["labels"])

# 2. 看每種實體各自的表現（PER / ORG / LOC / MISC）
#    在 compute_metrics 裡 print(results)，results 內有各類別的分數

# 3. 把被切開的詞合併，輸出更好讀
pipeline("token-classification", model="NER_Finetune_model",
         tokenizer="NER_Finetune_model", aggregation_strategy="simple")

# 4. 多測幾句，觀察錯誤類型
for s in ["Barack Obama visited Paris last week.",
          "Apple released a new iPhone in California."]:
    print(token_classifier(s))

# 5. 把 num_train_epochs 改成 3，比較 F1

# 6. 比較「只標第一塊子詞、其餘給 -100」的做法，看 F1 有沒有差異
```

## 八、三種任務的對照

| | 序列分類 | **Token 分類（NER）** | 遮蔽語言模型 |
|---|---|---|---|
| 例子 | 影評正負面 | 找人名、地名 | 填空 |
| 模型類別 | `...ForSequenceClassification` | `...ForTokenClassification` | `...ForMaskedLM` |
| 標籤粒度 | 句 | **token** | 被遮住的 token |
| 需要標籤對齊 | 不需要 | **需要** | 不需要 |
| 指標 | F1 | **seqeval 實體 F1** | loss / 困惑度 |

## 九、一句話記住

> **NER 微調的難點不在訓練，而在「標籤對齊」：把一個詞一個的標籤，依 `word_ids` 攤到一個 token 一個的位置，特殊 token 給 -100，被切開的 B 改成 I。**